# Conventional feature selection — feature-name tracking

This version records actual selected feature names for each inner CV fold, every outer-validation run, the final 455-observation fit, and the final held-out test evaluation. It also produces `level2_inner_cv_summary` (the requested name, although this is the conventional workflow).


In [14]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [15]:
%pip install ucimlrepo


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

### Explore the Data

In [17]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

((569, 30),
 (569,),
 None,
 None,
 Diagnosis
 B    357
 M    212
 Name: count, dtype: int64)

#### Check for duplicates

In [18]:
X.duplicated().sum()

np.int64(0)

In [19]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

### Load the shared agentic data partition

Use the saved Level 1 agentic partition manifest for the 455 development rows and reserved 114-row test set. The conventional development evaluation then loads the saved outer split manifest and evaluates the exact same train and validation row IDs.


In [20]:
from pathlib import Path
import json
import os

def _find_agentic_reference_dir():
    explicit_dir = os.environ.get("AGENTIC_REFERENCE_DIR")
    if explicit_dir:
        candidate = Path(explicit_dir).expanduser().resolve()
        if (candidate / "data_partition_manifest.json").is_file() and (candidate / "split_manifest.csv").is_file():
            return candidate
        raise FileNotFoundError(f"AGENTIC_REFERENCE_DIR is missing required manifest files: {candidate}")
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base / "Agentic" / "level_1", base / "level_1"])
    for candidate in candidates:
        if (candidate / "data_partition_manifest.json").is_file() and (candidate / "split_manifest.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find Agentic/level_1/data_partition_manifest.json and split_manifest.csv. "
        "Open this notebook from the Conventional project or set AGENTIC_REFERENCE_DIR explicitly."
    )

AGENTIC_REFERENCE_DIR = _find_agentic_reference_dir()
partition_manifest = json.loads(
    (AGENTIC_REFERENCE_DIR / "data_partition_manifest.json").read_text(encoding="utf-8")
)
development_row_ids = [int(value) for value in partition_manifest["development_row_ids"]]
reserved_test_row_ids = [int(value) for value in partition_manifest["reserved_test_row_ids"]]

if set(development_row_ids) & set(reserved_test_row_ids):
    raise ValueError("Agentic development and reserved test row IDs overlap.")
if set(development_row_ids) | set(reserved_test_row_ids) != set(X.index):
    raise ValueError("Agentic partition IDs do not match the loaded dataset row IDs.")

# Preserve the manifest's row order and keep the final test partition out of development selection.
X_train = X.loc[development_row_ids].copy()
y_train = y.loc[development_row_ids].copy()
X_test = X.loc[reserved_test_row_ids].copy()
y_test = y.loc[reserved_test_row_ids].copy()

print(f"Loaded shared agentic partition from: {AGENTIC_REFERENCE_DIR}")
print(f"Development rows: {len(X_train)} | Reserved test rows: {len(X_test)}")


Loaded shared agentic partition from: C:\Users\ogeuc\Downloads\Agentic\level_1
Development rows: 455 | Reserved test rows: 114


**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

### Define models, apply transformation, select features and evaluate the models

**Define Candidate Model Pipelines**

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Shared outer holdouts with five-fold inner CV


In [22]:
import time
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_SEEDS = [11, 42, 67]
REPEATS_PER_SEED = 3
CV_FOLDS = 5
SELECTION_METRIC = "F1 primary, Recall secondary"
ML_MODELS = {"LR": LR_pipeline, "SVC": SVC_pipeline, "RF": RF_pipeline, "GB": GB_pipeline, "XGB": XGB_pipeline}


In [23]:

shared_splits_df = pd.read_csv(AGENTIC_REFERENCE_DIR / "split_manifest.csv")
required_split_columns = {"Run_ID", "Seed", "Repeat", "Development_Train_Row_IDs", "Outer_Validation_Row_IDs"}
missing_columns = required_split_columns - set(shared_splits_df.columns)
if missing_columns:
    raise ValueError(f"Agentic split manifest is missing columns: {sorted(missing_columns)}")
if len(shared_splits_df) != len(RANDOM_SEEDS) * REPEATS_PER_SEED:
    raise ValueError("Expected exactly three repeats for each of the three agentic seeds.")
if set(shared_splits_df["Seed"].astype(int)) != set(RANDOM_SEEDS):
    raise ValueError("Agentic split manifest seeds do not match the expected seed list.")

def _row_ids(value):
    return [int(row_id) for row_id in json.loads(value)]

shared_splits_df = shared_splits_df.sort_values(["Seed", "Repeat", "Run_ID"]).reset_index(drop=True)
shared_splits = []
development_ids = set(X_train.index)
for split in shared_splits_df.to_dict("records"):
    train_ids = _row_ids(split["Development_Train_Row_IDs"])
    validation_ids = _row_ids(split["Outer_Validation_Row_IDs"])
    if set(train_ids) & set(validation_ids):
        raise ValueError(f"Train/validation overlap in agentic run {split['Run_ID']}.")
    if set(train_ids) | set(validation_ids) != development_ids:
        raise ValueError(f"Run {split['Run_ID']} does not partition the shared development rows.")
    shared_splits.append({**split, "train_ids": train_ids, "validation_ids": validation_ids})

for seed in RANDOM_SEEDS:
    seed_splits = [row for row in shared_splits if int(row["Seed"]) == seed]
    if len(seed_splits) != REPEATS_PER_SEED:
        raise ValueError(f"Seed {seed} does not have exactly three repeats.")
    reference = (seed_splits[0]["train_ids"], seed_splits[0]["validation_ids"])
    if any((row["train_ids"], row["validation_ids"]) != reference for row in seed_splits[1:]):
        raise ValueError(f"Repeated rows for seed {seed} do not reuse identical agentic indices.")

def _f1_then_recall_refit(cv_results):
    """Select highest mean CV F1, break ties by recall, then smaller k."""
    f1_scores = np.asarray(cv_results["mean_test_f1"], dtype=float)
    recall_scores = np.asarray(cv_results["mean_test_recall"], dtype=float)
    k_values = np.asarray([params["selector__k"] for params in cv_results["params"]])
    best_f1 = np.nanmax(f1_scores)
    f1_ties = np.flatnonzero(np.isclose(f1_scores, best_f1))
    best_recall = np.nanmax(recall_scores[f1_ties])
    finalists = f1_ties[np.isclose(recall_scores[f1_ties], best_recall)]
    return int(finalists[np.argmin(k_values[finalists])])


def feature_search(model, X_fit, y_fit, cv):
    pipeline = Pipeline([
        ("selector", SelectKBest(score_func=f_classif)),
        ("classifier", clone(model)),
    ])
    return GridSearchCV(
        estimator=pipeline,
        param_grid={"selector__k": list(range(1, X_fit.shape[1] + 1))},
        scoring={
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "auc": "roc_auc"
        },
        refit=_f1_then_recall_refit,
        cv=cv, n_jobs=-1, return_train_score=False,
    )

def _best_search_metrics(search):
    ix = int(search.best_index_)
    return {
        "Inner_CV_Recall": float(search.cv_results_["mean_test_recall"][ix]),
        "Inner_CV_Recall_SD": float(search.cv_results_["std_test_recall"][ix]),
        "Inner_CV_F1": float(search.cv_results_["mean_test_f1"][ix]),
        "Inner_CV_F1_SD": float(search.cv_results_["std_test_f1"][ix]),
        "Selected_k": int(search.best_params_["selector__k"]),
        "Candidate_Configurations": len(search.cv_results_["params"]),
        "CV_Fits": len(search.cv_results_["params"]) * search.n_splits_ + 1,
    }

def selected_feature_names(fitted_pipeline, original_columns):
    """Read the already-fitted SelectKBest mask; never fit on validation/test data."""
    selector = fitted_pipeline.named_steps["selector"]
    return list(pd.Index(original_columns)[selector.get_support()])


def inner_fold_feature_records(search, X_outer_train, y_outer_train, cv, run_id, seed, repeat, model_name):
    """Reconstruct ANOVA masks for each inner training fold at the winning k.

    This is diagnostic feature reporting, NOT another classifier fit or an
    independent CV evaluation. Feature names can differ between inner folds.
    """
    best_k = int(search.best_params_["selector__k"])
    records = []
    for fold, (inner_train_idx, inner_val_idx) in enumerate(cv.split(X_outer_train, y_outer_train), start=1):
        selector = SelectKBest(score_func=f_classif, k=best_k)
        selector.fit(X_outer_train.iloc[inner_train_idx], y_outer_train.iloc[inner_train_idx])
        records.append({
            "Run_ID": run_id, "Seed": seed, "Repeat": repeat,
            "Model": model_name, "Inner_Fold": fold,
            "Selected_k": best_k,
            "Selected_Features": selected_feature_names(
                Pipeline([("selector", selector)]), X_outer_train.columns
            ),
            "Inner_Train_Rows": len(inner_train_idx),
            "Inner_Validation_Rows": len(inner_val_idx),
        })
    return records


def score_validation(model, X_validation, y_validation):
    y_pred = model.predict(X_validation)
    if hasattr(model, "predict_proba"):
        positive_index = list(model.classes_).index(1)
        y_score = model.predict_proba(X_validation)[:, positive_index]
    else:
        y_score = model.decision_function(X_validation)
        if model.classes_[1] != 1:
            y_score = -y_score
    return {
        "Accuracy": accuracy_score(y_validation, y_pred),
        "Precision": precision_score(y_validation, y_pred, zero_division=0),
        "Recall": recall_score(y_validation, y_pred, zero_division=0),
        "F1": f1_score(y_validation, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_validation, y_score),
        "Confusion_matrix": confusion_matrix(y_validation, y_pred, labels=[0, 1]).tolist(),
    }


#### Inner feature selection uses five-fold CV; outer scoring uses the shared holdout rows


### Feature-name audit

The inner-CV report records the selected `k` and the actual feature names for each inner training fold, reconstructed at the winning `k`. This reconstruction is for auditing only and does not fit another classifier or affect the CV scores. Outer-validation features are taken from the search estimator refitted on the full outer-training subset. Final-development and test features are read from the same saved fitted pipeline; test labels are never used for selection.


In [24]:
# `feature_search` above ranks k by F1, then recall, with smaller k as a final deterministic tie-break.


In [25]:
candidate_cv_rows = []
inner_fold_feature_rows = []
repeated_results = []
for split in shared_splits:
    run_id, seed, repeat = int(split["Run_ID"]), int(split["Seed"]), int(split["Repeat"])
    X_outer_train = X.loc[split["train_ids"]]
    y_outer_train = y.loc[split["train_ids"]]
    X_outer_validation = X.loc[split["validation_ids"]]
    y_outer_validation = y.loc[split["validation_ids"]]
    inner_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    searches = {}
    run_candidate_rows = []
    run_started = time.perf_counter()

    for model_name, model in ML_MODELS.items():
        search = feature_search(model, X_outer_train, y_outer_train, inner_cv)
        search_started = time.perf_counter()
        search.fit(X_outer_train, y_outer_train)
        search_runtime = time.perf_counter() - search_started
        searches[model_name] = search
        details = _best_search_metrics(search)
        # Feature mask fitted on all outer-training rows (for outer holdout).
        outer_selected_features = selected_feature_names(
            search.best_estimator_, X_outer_train.columns
        )
        # Fold-specific feature masks are reconstructed on inner training only.
        inner_fold_feature_rows.extend(inner_fold_feature_records(
            search, X_outer_train, y_outer_train, inner_cv,
            run_id, seed, repeat, model_name
        ))
        candidate_row = {
            "Run_ID": run_id, "Seed": seed, "Repeat": repeat, "Model": model_name,
            "Runtime_seconds": float(search_runtime),
            "Selected_Features": outer_selected_features,
            **details,
        }
        candidate_cv_rows.append(candidate_row)
        run_candidate_rows.append(candidate_row)

    ranked = pd.DataFrame(run_candidate_rows).sort_values(
        ["Inner_CV_F1", "Inner_CV_Recall"], ascending=[False, False], kind="stable"
    )
    selected = ranked.iloc[0]
    selected_model_name = selected["Model"]
    selected_search = searches[selected_model_name]
    outer_metrics = score_validation(selected_search.best_estimator_, X_outer_validation, y_outer_validation)
    total_runtime = time.perf_counter() - run_started
    repeated_results.append({
        "Level": "Conventional feature selection", "Run_ID": run_id, "Seed": seed, "Repeat": repeat,
        "Split_Seed": seed, "Status": "completed", "Classifier": selected_model_name,
        "Feature_Method": "SelectKBest(f_classif)", "Selected_k": int(selected["Selected_k"]),
        "Selected_Features": selected_feature_names(
            selected_search.best_estimator_, X_outer_train.columns
        ),
        "Selection_Metric": SELECTION_METRIC,
        "Inner_CV_Recall": float(selected["Inner_CV_Recall"]), "Inner_CV_F1": float(selected["Inner_CV_F1"]),
        "CV_Folds": CV_FOLDS, "Model_Evaluations": int(sum(row["Candidate_Configurations"] for row in run_candidate_rows)),
        "Model_Fits": int(sum(row["CV_Fits"] for row in run_candidate_rows)),
        "Agent_Runtime": None, "Total_Run_Wall_Time": total_runtime,
        **{f"Outer_Validation_{key}": value for key, value in outer_metrics.items() if key != "Confusion_matrix"},
        "Outer_Validation_Confusion_Matrix": json.dumps(outer_metrics["Confusion_matrix"]),
        "Development_Train_Row_IDs": json.dumps(split["train_ids"]),
        "Outer_Validation_Row_IDs": json.dumps(split["validation_ids"]),
    })

candidate_cv_df = pd.DataFrame(candidate_cv_rows)
inner_fold_features_df = pd.DataFrame(inner_fold_feature_rows)
repeated_results_df = pd.DataFrame(repeated_results)


In [26]:
candidate_summary_df = (candidate_cv_df.groupby("Model", as_index=False)
    .agg(Inner_CV_Recall_Mean=("Inner_CV_Recall", "mean"), Inner_CV_Recall_SD=("Inner_CV_Recall", "std"),
         Inner_CV_F1_Mean=("Inner_CV_F1", "mean"), Inner_CV_F1_SD=("Inner_CV_F1", "std"),
         Selected_k_Mean=("Selected_k", "mean"), Selected_k_SD=("Selected_k", "std"),
         Runtime_Seconds_Mean=("Runtime_seconds", "mean"), Runtime_Seconds_SD=("Runtime_seconds", "std"))
    .sort_values(["Inner_CV_F1_Mean", "Inner_CV_Recall_Mean"], ascending=[False, False]).reset_index(drop=True))
outer_metrics = ["Outer_Validation_Accuracy", "Outer_Validation_Precision", "Outer_Validation_Recall", "Outer_Validation_F1", "Outer_Validation_AUC"]
outer_summary_df = pd.DataFrame([{
    "Outer_Runs": len(repeated_results_df),
    **{f"{metric}_Mean": float(repeated_results_df[metric].mean()) for metric in outer_metrics},
    **{f"{metric}_SD": float(repeated_results_df[metric].std(ddof=1)) for metric in outer_metrics},
    "Wall_Time_Seconds_Mean": float(repeated_results_df["Total_Run_Wall_Time"].mean()),
    "Candidate_Configurations_Mean": float(repeated_results_df["Model_Evaluations"].mean()),
    "Model_Fits_Mean": float(repeated_results_df["Model_Fits"].mean()),
}])
selection_consistency_df = (repeated_results_df.groupby("Classifier", as_index=False)
    .agg(Run_Count=("Run_ID", "count"), Seeds=("Seed", lambda values: sorted(set(map(int, values)))))
    .sort_values("Run_Count", ascending=False).reset_index(drop=True))
display(candidate_summary_df.round(4))
display(outer_summary_df.round(4))
display(selection_consistency_df)

# The requested Level 2-style inner-CV summary for this conventional notebook.
# One row per classifier, plus lists of the actual features chosen per run.
# The SD is across run-level inner-CV means, not independent fold scores.
level2_inner_cv_summary = candidate_summary_df.copy()
feature_lists_by_model = (
    candidate_cv_df.groupby("Model")["Selected_Features"]
    .apply(lambda series: [list(features) for features in series])
    .to_dict()
)
level2_inner_cv_summary["Selected_Features_By_Run"] = (
    level2_inner_cv_summary["Model"].map(feature_lists_by_model)
)

print("Inner-CV summary with selected feature names by run:")
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(level2_inner_cv_summary.round(4))

print("Inner-CV fold-specific feature selections:")
with pd.option_context("display.max_colwidth", None):
    display(inner_fold_features_df)

print("Outer-validation fitted feature selections:")
with pd.option_context("display.max_colwidth", None):
    display(repeated_results_df[[
        "Run_ID", "Seed", "Repeat", "Classifier", "Selected_k",
        "Selected_Features", "Outer_Validation_Recall", "Outer_Validation_F1"
    ]].round(4))


,Model,Inner_CV_Recall_Mean,Inner_CV_Recall_SD,Inner_CV_F1_Mean,Inner_CV_F1_SD,Selected_k_Mean,Selected_k_SD,Runtime_Seconds_Mean,Runtime_Seconds_SD
0,LR,0.9508,0.0096,0.9685,0.0051,21.6667,2.1794,1.8481,2.9541
1,SVC,0.9508,0.0099,0.9648,0.0037,23.0000,4.5000,0.8761,0.0404
2,XGB,0.9558,0.0128,0.9593,0.0039,19.3333,1.0000,9.0204,0.6413
3,GB,0.9335,0.0001,0.9444,0.0031,23.6667,5.0744,4.6480,0.1823
4,RF,0.9287,0.0038,0.9438,0.0035,22.6667,1.8028,4.6695,0.3127


,Outer_Runs,Outer_Validation_Accuracy_Mean,Outer_Validation_Precision_Mean,Outer_Validation_Recall_Mean,Outer_Validation_F1_Mean,Outer_Validation_AUC_Mean,Outer_Validation_Accuracy_SD,Outer_Validation_Precision_SD,Outer_Validation_Recall_SD,Outer_Validation_F1_SD,Outer_Validation_AUC_SD,Wall_Time_Seconds_Mean,Candidate_Configurations_Mean,Model_Fits_Mean
0,9,0.9634,0.9423,0.9608,0.9513,0.996,0.011,0.0016,0.0294,0.0151,0.0031,21.2325,150.0,755.0


,Classifier,Run_Count,Seeds
0,LR,6,"[42, 67]"
1,SVC,3,[11]


Inner-CV summary with selected feature names by run:


,Model,Inner_CV_Recall_Mean,Inner_CV_Recall_SD,Inner_CV_F1_Mean,Inner_CV_F1_SD,Selected_k_Mean,Selected_k_SD,Runtime_Seconds_Mean,Runtime_Seconds_SD,Selected_Features_By_Run
0,LR,0.9508,0.0096,0.9685,0.0051,21.6667,2.1794,1.8481,2.9541,"[[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3], [radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3], [radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3], [radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]]"
1,SVC,0.9508,0.0099,0.9648,0.0037,23.0000,4.5000,0.8761,0.0404,"[[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, fractal_dimension1, radius2, perimeter2, area2, smoothness2, compactness2, concavity2, concave_points2, symmetry2, fractal_dimension2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3], [radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, fractal_dimension1, radius2, perimeter2, area2, smoothness2, compactness2, concavity2, concave_points2, symmetry2, fractal_dimension2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, conca

Inner-CV fold-specific feature selections:


,Run_ID,Seed,Repeat,Model,Inner_Fold,Selected_k,Selected_Features,Inner_Train_Rows,Inner_Validation_Rows
0,1,11,1,LR,1,24,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",291,73
1,1,11,1,LR,2,24,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",291,73
2,1,11,1,LR,3,24,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",291,73
3,1,11,1,LR,4,24,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",291,73
4,1,11,1,LR,5,24,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",292,72
...,...,...,...,...,...,...,...,...,...
220,9,67,3,XGB,1,18,"[radius1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3]",291,73
221,9,67,3,XGB,2,18,"[radius1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3]",291,73
222,9,67,3,XGB,3,18,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3]",291,73
223,9,67,3,XGB,4,18,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3]",291,73


Outer-validation fitted feature selections:


,Run_ID,Seed,Repeat,Classifier,Selected_k,Selected_Features,Outer_Validation_Recall,Outer_Validation_F1
0,1,11,1,SVC,20,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",1.0000,0.9714
1,2,11,2,SVC,20,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",1.0000,0.9714
2,3,11,3,SVC,20,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",1.0000,0.9714
3,4,42,1,LR,19,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",0.9412,0.9412
4,5,42,2,LR,19,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",0.9412,0.9412
5,6,42,3,LR,19,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",0.9412,0.9412
6,7,67,1,LR,22,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",0.9412,0.9412
7,8,67,2,LR,22,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",0.9412,0.9412
8,9,67,3,LR,22,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",0.9412,0.9412


In [27]:
# Candidate and outer-holdout summaries are displayed in the preceding cell.


In [28]:
print("Outer holdout runs:", len(repeated_results_df))
print("Seeds:", RANDOM_SEEDS, "| repeats per seed:", REPEATS_PER_SEED)
print("Selection rule: highest inner-CV F1, then recall; smaller k breaks remaining ties.")


Outer holdout runs: 9
Seeds: [11, 42, 67] | repeats per seed: 3
Selection rule: highest inner-CV F1, then recall; smaller k breaks remaining ties.


In [29]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [30]:
# Select the final model and feature count on all development data only; do not read X_test or y_test here.
final_selection_rows = []
final_searches = {}
final_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=101)
final_fit_start = time.perf_counter()
final_fit_cpu_start = time.process_time()
for model_name, model in ML_MODELS.items():
    search = feature_search(model, X_train, y_train, final_cv)

    model_start = time.perf_counter()
    search.fit(X_train, y_train)
    model_runtime = time.perf_counter() - model_start
    
    best_idx = int(search.best_index_)

    # Additional five-fold CV metrics
    extra_metrics = {}

    for metric, label in [
        ("accuracy", "Accuracy"),
        ("precision", "Precision"),
        ("recall", "Recall"),
        ("f1", "F1"),
        ("auc", "AUC")
    ]:
        extra_metrics[f"{label}_Mean"] = float(
            search.cv_results_[f"mean_test_{metric}"][best_idx]
        )

        extra_metrics[f"{label}_SD"] = float(
            search.cv_results_[f"std_test_{metric}"][best_idx]
            )

        # Computational cost
        candidates = len(search.cv_results_["params"])
        cv_folds = search.n_splits_

        cv_model_fits = candidates * cv_folds
        development_model_fits = cv_model_fits + 1

    final_searches[model_name] = search
    final_selection_rows.append({
        "Model": model_name,
        **_best_search_metrics(search),
        "Selected_Features": selected_feature_names(
            search.best_estimator_, X_train.columns
        ),
        **extra_metrics,
        "CV_Search_Runtime_Seconds": model_runtime,
        "CV_Search_Candidates": candidates,
        "CV_Model_Fits": cv_model_fits,
        "Development_Model_Fits": development_model_fits
    })
final_model_selection_df = pd.DataFrame(final_selection_rows).sort_values(
    ["Inner_CV_F1", "Inner_CV_Recall"], ascending=[False, False], kind="stable"
).reset_index(drop=True)
best_model_name = final_model_selection_df.iloc[0]["Model"]
final_search = final_searches[best_model_name]
final_model = final_search.best_estimator_
final_k = int(final_search.best_params_["selector__k"])
selected_features = selected_feature_names(final_model, X_train.columns)
final_development_fit_seconds = time.perf_counter() - final_fit_start
final_fit_cpu_seconds = time.process_time() - final_fit_cpu_start

folder_name = "eval - feature selection"
output_dir = Path.cwd() if Path.cwd().name == folder_name else Path.cwd() / folder_name
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "conventional_frozen_model.joblib"
model_save_start = time.perf_counter(); joblib.dump(final_model, model_path); model_save_seconds = time.perf_counter() - model_save_start
model_load_start = time.perf_counter(); loaded_best_pipeline = joblib.load(model_path); model_load_seconds = time.perf_counter() - model_load_start
final_workflow_wall_seconds = final_development_fit_seconds + model_save_seconds + model_load_seconds


selected_final_cv = final_model_selection_df.iloc[0]
best_record = {
    "Best_Model": best_model_name, "Selection_Metric": SELECTION_METRIC,
    "Random_Seeds": RANDOM_SEEDS, "CV_Folds": CV_FOLDS,
    "Repeated_Outer_Holdouts_Per_Seed": REPEATS_PER_SEED, "Outer_Run_Count": len(shared_splits),
    "Final_Selected_Feature_Count": final_k, "Selected_Features": selected_features,
    "Final_Development_CV_Recall": float(selected_final_cv["Inner_CV_Recall"]),
    "Final_Development_CV_F1": float(selected_final_cv["Inner_CV_F1"]),
    "Final_Development_CV_Recall_SD": float(selected_final_cv["Inner_CV_Recall_SD"]),
    "Final_Development_CV_F1_SD": float(selected_final_cv["Inner_CV_F1_SD"]),
    "Final_Development_Fit_Runtime_Seconds": final_development_fit_seconds,
    "Final_Fit_CPU_Seconds": final_fit_cpu_seconds,
    "Model_Save_Runtime_Seconds": model_save_seconds, "Model_Load_Runtime_Seconds": model_load_seconds,
    "Final_Test_Scoring_Wall_Seconds": None, "Final_Test_Scoring_CPU_Seconds": None,
    "Final_Workflow_Wall_Seconds": final_workflow_wall_seconds,
    "Test_Scored": False, "Saved_Model_Path": str(model_path),
    "Shared_Agentic_Split_Manifest": str(AGENTIC_REFERENCE_DIR / "split_manifest.csv"),
}

shared_splits_df.to_csv(output_dir / "split_manifest.csv", index=False)
(output_dir / "data_partition_manifest.json").write_text(
    (AGENTIC_REFERENCE_DIR / "data_partition_manifest.json").read_text(encoding="utf-8"), encoding="utf-8")
for seed in RANDOM_SEEDS:
    repeated_results_df.loc[repeated_results_df["Seed"] == seed].to_csv(output_dir / f"conventional_seed_{seed}_runs.csv", index=False)
seed_selections = repeated_results_df[["Run_ID", "Seed", "Repeat", "Classifier", "Selected_k", "Selected_Features", "Inner_CV_Recall", "Inner_CV_F1",
    "Outer_Validation_Recall", "Outer_Validation_F1"]].rename(columns={"Classifier": "Selected_Model"})
seed_selections.to_csv(output_dir / "conventional_seed_selections.csv", index=False)
selection_consistency_df.to_csv(output_dir / "decision_consistency.csv", index=False)
candidate_summary_df.to_csv(output_dir / "all_pipelines_summary.csv", index=False)
candidate_summary_df.to_csv(output_dir / "conventional_pipeline_summary.csv", index=False)
candidate_cv_df.to_csv(output_dir / "candidate_inner_cv_results.csv", index=False)
inner_fold_features_df.to_csv(output_dir / "inner_cv_fold_selected_features.csv", index=False)
level2_inner_cv_summary.to_csv(output_dir / "level2_inner_cv_summary.csv", index=False)
final_model_selection_df.to_csv(output_dir / "final_model_selection_cv.csv", index=False)
outer_summary_df.to_csv(output_dir / "conventional_summary.csv", index=False)
outer_summary_df.to_csv(output_dir / "outer_holdout_summary.csv", index=False)
repeated_results_df.to_csv(output_dir / "conventional_repeated_runs.csv", index=False)

run_metadata = {
    "Workflow": "conventional_feature_selection_shared_outer_holdout_evaluation",
    "Random_Seeds": RANDOM_SEEDS, "CV_Folds": CV_FOLDS,
    "Repeated_Outer_Holdouts_Per_Seed": REPEATS_PER_SEED, "Outer_Run_Count": len(shared_splits),
    "Outer_Split_Source": str(AGENTIC_REFERENCE_DIR / "split_manifest.csv"),
    "Data_Partition_Source": str(AGENTIC_REFERENCE_DIR / "data_partition_manifest.json"),
    "Development_Row_Count": len(X_train), "Reserved_Test_Row_Count": len(X_test),
    "Selection_Metric": SELECTION_METRIC, "Feature_Count_Tie_Break": "smaller k",
    "Inner_CV_Uses_Outer_Training_Rows_Only": True, "Test_Scored": False,
}
(output_dir / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
frozen_workflow = {
    "Workflow_State": "development_fit_complete_test_pending", "Selected_Model": best_model_name,
    "Selected_Feature_Count": final_k, "Selected_Features": selected_features, "Saved_Model_Path": str(model_path),
    "Selection_Rule": SELECTION_METRIC, "Random_Seeds": RANDOM_SEEDS,
    "Repeated_Outer_Holdouts_Per_Seed": REPEATS_PER_SEED,
    "Split_Manifest": str(output_dir / "split_manifest.csv"),
    "Shared_Agentic_Split_Manifest": str(AGENTIC_REFERENCE_DIR / "split_manifest.csv"),
    "Run_Metadata": str(output_dir / "run_metadata.json"),
}
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps(frozen_workflow, indent=2), encoding="utf-8")
(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected final classifier and k using development CV F1, then recall:", best_model_name, final_k)
print("Final 455-row selected features:", selected_features)
with pd.option_context("display.max_colwidth", None):
    display(
    final_model_selection_df[[
        "Model",
        "Selected_k",
        "Selected_Features",

        "Accuracy_Mean",
        "Accuracy_SD",

        "Precision_Mean",
        "Precision_SD",

        "Recall_Mean",
        "Recall_SD",

        "F1_Mean",
        "F1_SD",

        "AUC_Mean",
        "AUC_SD",

        "CV_Search_Runtime_Seconds",
        "CV_Search_Candidates",
        "CV_Model_Fits",
        "Development_Model_Fits"
    ]].round(4)
)
print("Saved final development model:", model_path)
print("The reserved test set was not scored.")


Selected final classifier and k using development CV F1, then recall: XGB 28
Final 455-row selected features: ['radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 'concave_points1', 'symmetry1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 'compactness2', 'concavity2', 'concave_points2', 'fractal_dimension2', 'radius3', 'texture3', 'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 'fractal_dimension3']


,Model,Selected_k,Selected_Features,Accuracy_Mean,Accuracy_SD,Precision_Mean,Precision_SD,Recall_Mean,Recall_SD,F1_Mean,F1_SD,AUC_Mean,AUC_SD,CV_Search_Runtime_Seconds,CV_Search_Candidates,CV_Model_Fits,Development_Model_Fits
0,XGB,28,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, texture2, perimeter2, area2, smoothness2, compactness2, concavity2, concave_points2, fractal_dimension2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",0.9736,0.0088,0.9706,0.0181,0.9588,0.0144,0.9645,0.0118,0.9946,0.0034,9.8537,30,150,151
1,LR,29,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, texture2, perimeter2, area2, smoothness2, compactness2, concavity2, concave_points2, symmetry2, fractal_dimension2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",0.9736,0.0164,0.9765,0.0289,0.9529,0.0235,0.9643,0.0222,0.9955,0.0051,0.9842,30,150,151
2,SVC,19,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",0.9714,0.0226,0.9596,0.0345,0.9647,0.0343,0.9619,0.0303,0.9942,0.0053,0.8718,30,150,151
3,RF,19,"[radius1, texture1, perimeter1, area1, compactness1, concavity1, concave_points1, radius2, perimeter2, area2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3]",0.9604,0.0164,0.9533,0.0284,0.9412,0.0372,0.9466,0.0228,0.9876,0.0075,4.5784,30,150,151
4,GB,25,"[radius1, texture1, perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, radius2, perimeter2, area2, compactness2, concavity2, concave_points2, radius3, texture3, perimeter3, area3, smoothness3, compactness3, concavity3, concave_points3, symmetry3, fractal_dimension3]",0.9582,0.0082,0.9478,0.0264,0.9412,0.0186,0.9440,0.0104,0.9908,0.0061,5.6221,30,150,151


Saved final development model: c:\Users\ogeuc\Downloads\Conventional\eval - feature selection\conventional_frozen_model.joblib
The reserved test set was not scored.


### Final Development Fit and Saved Model (test scoring pending)

The saved estimator was selected and fit using development data only.


### Final Test Set Evaluation (one-time, after the workflow is frozen)

This is the only cell that scores the reserved test set. Do not use its result to modify code, settings, features, thresholds, or model choices.


In [31]:
# Run this cell only when ready to evaluate the held-out test set.
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

loaded_best_pipeline = joblib.load(model_path)
test_wall_start = time.perf_counter()
test_cpu_start = time.process_time()
y_test_pred = loaded_best_pipeline.predict(X_test)
if hasattr(loaded_best_pipeline, "predict_proba"):
    class_index = list(loaded_best_pipeline.classes_).index(1)
    y_test_score = loaded_best_pipeline.predict_proba(X_test)[:, class_index]
else:
    y_test_score = loaded_best_pipeline.decision_function(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_auc = roc_auc_score(y_test, y_test_score)
test_scoring_cpu_seconds = time.process_time() - test_cpu_start
test_scoring_seconds = time.perf_counter() - test_wall_start

best_record.update({
    "Final_Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Final_Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Test_Scored": True,
    "Final_Test_Accuracy": test_accuracy,
    "Final_Test_Precision": test_precision,
    "Final_Test_Recall": test_recall,
    "Final_Test_F1": test_f1,
    "Final_Test_ROC_AUC": test_auc,
    "Final_Test_Confusion_Matrix": confusion_matrix(y_test, y_test_pred).tolist()
})
test_features = selected_feature_names(loaded_best_pipeline, X_train.columns)
if test_features != selected_features:
    raise RuntimeError("Saved model feature mask differs from final training selection.")
best_record["Final_Test_Selected_Features"] = test_features
best_record["Final_Test_Selected_k"] = len(test_features)
test_metrics = {
    "Model": best_model_name,
    "Selected_k": len(test_features),
    "Selected_Features": test_features,
    "Test_Accuracy": test_accuracy,
    "Test_Precision": test_precision,
    "Test_Recall": test_recall,
    "Test_F1": test_f1,
    "Test_ROC_AUC": test_auc,
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds
}
pd.DataFrame([test_metrics]).to_csv(output_dir / "conventional_final_test.csv", index=False)
(output_dir / "conventional_frozen_workflow.json").write_text(json.dumps({
    "Workflow_State": "test_scoring_complete",
    "Selected_Model": best_model_name,
    "Saved_Model_Path": str(model_path),
    "Selected_Features": test_features,
    "Selected_k": len(test_features),
    "Test_Metrics_Path": str(output_dir / "conventional_final_test.csv"),
    "Test_Scoring_Wall_Seconds": test_scoring_seconds,
    "Test_Scoring_CPU_Seconds": test_scoring_cpu_seconds,
    "Random_Seeds": RANDOM_SEEDS
}, indent=2), encoding="utf-8")
(output_dir / "conventional_final_run.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
pd.DataFrame([best_record]).to_csv(output_dir / "conventional_final_run.csv", index=False)

(output_dir / "best_pipeline_record.json").write_text(json.dumps(best_record, indent=2), encoding="utf-8")
print("Selected pipeline:", best_model_name)
print("Selected feature count:", final_k)
print("Features applied to final test set:", test_features)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test ROC AUC: {test_auc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification report:\n", classification_report(y_test, y_test_pred, zero_division=0))


Selected pipeline: XGB
Selected feature count: 28
Features applied to final test set: ['radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 'concave_points1', 'symmetry1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 'compactness2', 'concavity2', 'concave_points2', 'fractal_dimension2', 'radius3', 'texture3', 'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 'fractal_dimension3']
Test accuracy: 0.9649
Test precision: 1.0000
Test recall: 0.9048
Test F1: 0.9500
Test ROC AUC: 0.9907
Confusion matrix:
 [[72  0]
 [ 4 38]]

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

